# Notebook 04 — Game Theory Prototype & Monte Carlo Uncertainty

**Stage D:** Two-household normal-form game with Nash equilibrium analysis  
**Stage E:** Monte Carlo uncertainty quantification and expected payoff estimation

**Selected hour:** `2013-06-08 10:00` — an hour in the held-out test period where:  
- H001 has substantial solar **surplus** (actual: +1.245 kWh)  
- H002 has substantial demand **deficit** (actual: −1.463 kWh)  

> **Tariff note.** All prices come from the *hypothetical illustrative* scenarios in `energy_price_scenarios.csv`. They are not observed market prices.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.energy.energy_balance import compute_energy_balance
from src.energy.grid import GridCostCalculator
from src.forecasting.predict import EnergyForecaster
from src.game.game import run_two_household_game
from src.game.nash import find_pure_nash_equilibria
from src.game.payoff_matrix import build_two_household_payoff_matrix
from src.stochastic.monte_carlo import (
    MonteCarloSimulator, ValidationResidualPool, estimate_expected_payoff_matrix
)
from src.utils.seed import set_seed

set_seed(42)
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Imports OK')

## Stage D — Two-household game

In [ ]:
# ── D1: Select the representative hour and compute energy balances ────────────
GAME_TS = '2013-06-08 10:00:00'
SCENARIO_ID = 'HYPOTHETICAL_BASE'

forecaster = EnergyForecaster(
    data_path=PROJECT_ROOT / 'data' / 'processed' / 'three_house_hourly_lstm_ready.csv',
    model_path=PROJECT_ROOT / 'data' / 'processed' / 'lstm_model.keras',
    scaler_path=PROJECT_ROOT / 'data' / 'processed' / 'lstm_scaler.joblib',
)

grid_calc = GridCostCalculator(PROJECT_ROOT / 'data' / 'processed' / 'energy_price_scenarios.csv')
scenario = grid_calc.get_scenario(SCENARIO_ID)

# LSTM forecasts for the game hour
pred_h1 = forecaster.predict_next_hour('H001', GAME_TS)
pred_h2 = forecaster.predict_next_hour('H002', GAME_TS)

# Energy balances from ACTUAL readings (ground truth)
b1_actual = compute_energy_balance(
    demand_kwh=pred_h1.actual_demand_kwh,
    solar_generation_kwh=pred_h1.actual_solar_generation_kwh,
    house_id='H001',
)
b2_actual = compute_energy_balance(
    demand_kwh=pred_h2.actual_demand_kwh,
    solar_generation_kwh=pred_h2.actual_solar_generation_kwh,
    house_id='H002',
)

print(f'Representative hour: {GAME_TS}')
print(f'H001 (SELLER): demand={b1_actual.demand_kwh:.3f} kWh, '
      f'solar={b1_actual.solar_generation_kwh:.3f} kWh, '
      f'SURPLUS={b1_actual.surplus_kwh:.3f} kWh')
print(f'H002 (BUYER):  demand={b2_actual.demand_kwh:.3f} kWh, '
      f'solar={b2_actual.solar_generation_kwh:.3f} kWh, '
      f'DEFICIT={b2_actual.deficit_kwh:.3f} kWh')

In [ ]:
# ── D2: Game setup — actions and constraints ──────────────────────────────────
print('=== GAME SETUP ===')
print(f'Player 1: H001 (Seller) | Available surplus: {b1_actual.surplus_kwh:.4f} kWh')
print(f'  Actions: OFFER_P2P (offer surplus to H002) | EXPORT_GRID (feed-in at {scenario.solar_export_credit_aud_per_kwh} AUD/kWh)')
print()
print(f'Player 2: H002 (Buyer) | Required deficit: {b2_actual.deficit_kwh:.4f} kWh')
print(f'  Actions: ACCEPT_P2P (buy from H001) | GRID_IMPORT (buy from grid at {scenario.import_price_aud_per_kwh} AUD/kWh)')
print()

# Mid-market P2P price: (P_buy + P_sell) / 2
p2p_price = (scenario.import_price_aud_per_kwh + scenario.solar_export_credit_aud_per_kwh) / 2
max_trade  = min(b1_actual.surplus_kwh, b2_actual.deficit_kwh)
print(f'P2P negotiated price: ({scenario.import_price_aud_per_kwh} + {scenario.solar_export_credit_aud_per_kwh}) / 2 = {p2p_price:.4f} AUD/kWh')
print(f'Maximum feasible trade: min({b1_actual.surplus_kwh:.4f}, {b2_actual.deficit_kwh:.4f}) = {max_trade:.4f} kWh')
print()
print('Energy constraints:')
print('  Seller cannot sell more than their surplus')
print('  Buyer cannot buy more than their deficit')
print('  Buyer payment = Seller revenue (transaction conservation)')

In [ ]:
# ── D3: Build and display payoff matrix ──────────────────────────────────────
matrix, eq_result, comp = run_two_household_game(
    b1_actual, b2_actual,
    grid_import_price_aud=scenario.import_price_aud_per_kwh,
    grid_export_credit_aud=scenario.solar_export_credit_aud_per_kwh,
)

print('=== PAYOFF MATRIX (U_H001, U_H002) in AUD ===')
print(f'Utility U_i = −(Net Expenditure_i) | Positive = net gain, Negative = net cost')
display(matrix.to_dataframe())

In [ ]:
# ── D4: Visualise the payoff matrix as a heatmap ─────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

for ax, title, payoffs in [
    (ax1, 'H001 Utility (AUD)', matrix.payoffs_p1),
    (ax2, 'H002 Utility (AUD)', matrix.payoffs_p2),
]:
    im = ax.imshow(payoffs, cmap='RdYlGn', aspect='auto',
                   vmin=min(payoffs.min(), -0.01))
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(matrix.actions_p2, fontsize=10)
    ax.set_yticklabels(matrix.actions_p1, fontsize=10)
    ax.set_xlabel('H002 Action')
    ax.set_ylabel('H001 Action')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.85)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{payoffs[i, j]:.4f}',
                    ha='center', va='center', fontsize=11, fontweight='bold')

plt.suptitle('Normal-Form Game Payoff Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'data' / 'processed' / '04_payoff_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── D5: Nash equilibrium report ───────────────────────────────────────────────
print('=== NASH EQUILIBRIUM ANALYSIS ===')
print(f'Pure-strategy Nash equilibria found: {eq_result.num_equilibria}')
print()

for k, (a2_name, br_a1_list) in enumerate(eq_result.best_responses_p1.items()):
    print(f'  Best response of H001 when H002 plays {a2_name}: {br_a1_list}')
for k, (a1_name, br_a2_list) in enumerate(eq_result.best_responses_p2.items()):
    print(f'  Best response of H002 when H001 plays {a1_name}: {br_a2_list}')
print()

if eq_result.has_pure_nash:
    for eq in eq_result.equilibria:
        print(f'  Equilibrium: H001={eq.action_player1}, H002={eq.action_player2}')
        print(f'    Utility H001 = {eq.utility_player1:.4f} AUD | Utility H002 = {eq.utility_player2:.4f} AUD')
        print(f'    Energy traded = {eq.traded_energy_kwh:.4f} kWh | Strict: {eq.is_strict}')
        print()
else:
    print('  No pure-strategy Nash equilibrium exists.')

In [ ]:
# ── D6: Baseline vs Game comparison table ────────────────────────────────────
print('=== BASELINE (EXPORT_GRID, GRID_IMPORT) vs GAME EQUILIBRIUM ===')
print()
display(comp.to_dataframe())

print()
if comp.is_nash_equilibrium:
    print(f'Community savings (Nash eq. vs uncoordinated baseline): '
          f'{comp.total_community_savings_aud:.4f} AUD')
    print(f'  H001 savings: {comp.saving_p1_aud:.4f} AUD  '
          f'| H002 savings: {comp.saving_p2_aud:.4f} AUD')
    print(f'Energy traded via P2P: {comp.energy_traded_kwh:.4f} kWh at {comp.p2p_price_aud:.4f} AUD/kWh')
    print(f'Equilibrium type: {"STRICT" if comp.is_strict_equilibrium else "WEAK"}')
else:
    print('No Nash equilibrium — no coordination gain to report.')

## Stage E — Monte Carlo uncertainty quantification

In [ ]:
# ── E1: Load validation residuals and compute statistics ─────────────────────
residual_pool = ValidationResidualPool(
    forecaster=forecaster,
    sequences_path=PROJECT_ROOT / 'data' / 'processed' / 'lstm_sequences.npz',
)

print('=== VALIDATION RESIDUAL STATISTICS ===')
print('Source: chronological validation set (March–April 2013, 2,103 hours per house)')
print('Residual: e_t = y_t − ŷ_t  (actual − predicted) in kWh')
print()

for house in ['H001', 'H002', 'H003']:
    d_stat, s_stat = residual_pool.get_statistics(house_id=house)
    print(f'{house} Demand residuals  (N={d_stat.sample_size}): '
          f'mean={d_stat.mean:+.4f}, σ={d_stat.std_dev:.4f} kWh, '
          f'[Q5={d_stat.q05:.3f}, Q95={d_stat.q95:.3f}]')
    print(f'{house} Solar residuals   (N={s_stat.sample_size}): '
          f'mean={s_stat.mean:+.4f}, σ={s_stat.std_dev:.4f} kWh, '
          f'[Q5={s_stat.q05:.3f}, Q95={s_stat.q95:.3f}]')
    print()

In [ ]:
# ── E2: Generate 1,000 scenarios for H001 and H002 ───────────────────────────
mc_sim = MonteCarloSimulator(residual_pool=residual_pool)

dist_h1 = mc_sim.generate_scenarios(pred_h1, num_scenarios=1000, random_seed=42, sampling_method='empirical')
dist_h2 = mc_sim.generate_scenarios(pred_h2, num_scenarios=1000, random_seed=42, sampling_method='empirical')

print('=== 1,000-SCENARIO DISTRIBUTION SUMMARY ===')
print('Method: Empirical resampling of paired validation residuals (preserves demand–solar correlation)')
print(f'Physical bounds enforced: demand ≥ 0, solar ≥ 0')
print()

print('H001:')
display(dist_h1.get_summary_dataframe())
print()
print('H002:')
display(dist_h2.get_summary_dataframe())

In [ ]:
# ── E3: Visualise scenario distributions ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

plot_configs = [
    (axes[0, 0], dist_h1.scenarios_demand_kwh, dist_h1.forecast_demand_mean_kwh,
     dist_h1.actual_demand_kwh, 'H001 Demand (kWh)', '#e74c3c'),
    (axes[0, 1], dist_h1.scenarios_solar_kwh,  dist_h1.forecast_solar_mean_kwh,
     dist_h1.actual_solar_kwh,  'H001 Solar (kWh)',  '#f39c12'),
    (axes[1, 0], dist_h2.scenarios_demand_kwh, dist_h2.forecast_demand_mean_kwh,
     dist_h2.actual_demand_kwh, 'H002 Demand (kWh)', '#e74c3c'),
    (axes[1, 1], dist_h2.scenarios_solar_kwh,  dist_h2.forecast_solar_mean_kwh,
     dist_h2.actual_solar_kwh,  'H002 Solar (kWh)',  '#f39c12'),
]

for ax, scenarios, forecast_mean, actual, title, color in plot_configs:
    ax.hist(scenarios, bins=40, color=color, alpha=0.6, density=True, label='Scenarios')
    ax.axvline(forecast_mean, color='navy',  linewidth=2, linestyle='--', label=f'Forecast: {forecast_mean:.3f}')
    if actual is not None:
        ax.axvline(actual, color='black', linewidth=2, label=f'Actual: {actual:.3f}')
    q5, q95 = np.percentile(scenarios, [5, 95])
    ax.axvspan(q5, q95, alpha=0.15, color=color, label=f'90% PI: [{q5:.2f}, {q95:.2f}]')
    ax.set_xlabel('Energy (kWh)')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle(f'1,000 Monte Carlo Scenarios — {GAME_TS}\n(Empirical residual resampling from validation set)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'data' / 'processed' / '04_monte_carlo_scenarios.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── E4: Expected payoff matrix under uncertainty ──────────────────────────────
exp_df, exp_u1, exp_u2 = estimate_expected_payoff_matrix(
    dist_h1, dist_h2,
    grid_import_price_aud=scenario.import_price_aud_per_kwh,
    grid_export_credit_aud=scenario.solar_export_credit_aud_per_kwh,
)

print('=== EXPECTED PAYOFF MATRIX UNDER 1,000 MONTE CARLO SCENARIOS ===')
print('Note: E[U_i(a)] computed by averaging payoffs across all scenarios.')
print('This is NOT averaging 1,000 independent Nash equilibria.')
print()
display(exp_df)

In [ ]:
# ── E5: Deterministic vs stochastic comparison ────────────────────────────────
print('=== DETERMINISTIC vs STOCHASTIC PAYOFF COMPARISON ===')
print()

det_u1 = matrix.payoffs_p1
det_u2 = matrix.payoffs_p2

action_pairs = [
    (0, 0, 'OFFER_P2P / ACCEPT_P2P'),
    (0, 1, 'OFFER_P2P / GRID_IMPORT'),
    (1, 0, 'EXPORT_GRID / ACCEPT_P2P'),
    (1, 1, 'EXPORT_GRID / GRID_IMPORT (Baseline)'),
]

comparison_rows = []
for i, j, label in action_pairs:
    comparison_rows.append({
        'Action Profile': label,
        'Det U_H001': round(det_u1[i, j], 4),
        'Exp U_H001': round(exp_u1[i, j], 4),
        'Det U_H002': round(det_u2[i, j], 4),
        'Exp U_H002': round(exp_u2[i, j], 4),
        'Det Sum': round(det_u1[i, j] + det_u2[i, j], 4),
        'Exp Sum': round(exp_u1[i, j] + exp_u2[i, j], 4),
    })
display(pd.DataFrame(comparison_rows))

In [ ]:
# ── E6: Limitations and next steps ───────────────────────────────────────────
print('=== LIMITATIONS AND NEXT STEPS ===')
print()
print('Current prototype limitations:')
print('  1. Only 2 of 3 households participate in the game (H001 & H002).')
print('  2. P2P price is fixed at mid-market; no bilateral negotiation mechanism.')
print('  3. Validation residuals cover only March–April 2013; seasonal variation not captured.')
print('  4. H002 residuals show systematic bias (mean demand residual: −0.30 kWh),')
print('     meaning the LSTM under-predicts H002 demand — scenarios may under-estimate deficit.')
print('  5. Monte Carlo scenarios not yet connected to a stochastic Nash equilibrium solver.')
print('  6. No battery storage in the energy balance (excluded by design to avoid incorrect accounting).')
print()
print('Next-phase tasks:')
print('  - Extend game to all 3 households and multi-hour horizons.')
print('  - Add coalition formation and social welfare analysis.')
print('  - Connect expected payoff matrix to robust / stochastic Nash equilibrium.')
print('  - Replace hypothetical tariffs with verified retailer prices before economic interpretation.')
print('  - Improve LSTM demand forecast (currently worse than persistence baseline on demand).')

In [ ]:
# ── Summary report ────────────────────────────────────────────────────────────
print('=' * 65)
print('PROTOTYPE RESULTS SUMMARY')
print('=' * 65)
print(f'Representative hour:      {GAME_TS}')
print(f'Tariff scenario:          {scenario.scenario_label} (HYPOTHETICAL)')
print(f'Grid import price:        {scenario.import_price_aud_per_kwh} AUD/kWh')
print(f'Grid export credit:       {scenario.solar_export_credit_aud_per_kwh} AUD/kWh')
print(f'P2P negotiated price:     {comp.p2p_price_aud:.4f} AUD/kWh (mid-market)')
print()
print('Energy at selected hour (ACTUAL readings):')
print(f'  H001: demand={b1_actual.demand_kwh:.3f} kWh, solar={b1_actual.solar_generation_kwh:.3f} kWh, '
      f'surplus={b1_actual.surplus_kwh:.3f} kWh')
print(f'  H002: demand={b2_actual.demand_kwh:.3f} kWh, solar={b2_actual.solar_generation_kwh:.3f} kWh, '
      f'deficit={b2_actual.deficit_kwh:.3f} kWh')
print()
print('Baseline (EXPORT_GRID, GRID_IMPORT):')
print(f'  H001 net cost: {comp.baseline_cost_p1_aud:.4f} AUD (negative = net gain from feed-in)')
print(f'  H002 net cost: {comp.baseline_cost_p2_aud:.4f} AUD')
print(f'  Community total: {comp.baseline_total_cost_aud:.4f} AUD')
print()
if comp.is_nash_equilibrium:
    print(f'Nash Equilibrium: ({comp.selected_strategy_p1}, {comp.selected_strategy_p2})')
    print(f'  Energy traded via P2P: {comp.energy_traded_kwh:.4f} kWh')
    print(f'  H001 net cost: {comp.game_cost_p1_aud:.4f} AUD | Saving: {comp.saving_p1_aud:.4f} AUD')
    print(f'  H002 net cost: {comp.game_cost_p2_aud:.4f} AUD | Saving: {comp.saving_p2_aud:.4f} AUD')
    print(f'  Community total: {comp.game_total_cost_aud:.4f} AUD | '
          f'Total saving: {comp.total_community_savings_aud:.4f} AUD')
else:
    print('No pure-strategy Nash equilibrium found.')
print()
print('Monte Carlo (1,000 scenarios, empirical residual resampling):')
d_stat, _ = residual_pool.get_statistics(house_id='H001')
_, s_stat = residual_pool.get_statistics(house_id='H001')
print(f'  H001 demand forecast: {pred_h1.predicted_demand_kwh:.4f} kWh | '
      f'Residual σ = {d_stat.std_dev:.4f} kWh')
print(f'  H001 actual: {pred_h1.actual_demand_kwh:.4f} kWh | '
      f'In 90% PI: {dist_h1.get_summary_dataframe().iloc[0]["In_90pct_PI"]}')
print('=' * 65)